# 🏗️ Notebook 1: Airbnb — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/airbnb
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A property-rental marketplace. Guests search listings by location & date, see availability, book, and pay. Hosts list properties, set prices, and get paid out.

We care about two hard problems:

1. **Search**: fast geo + date filters across millions of listings.
2. **Booking**: no double-bookings — even when two guests click *Reserve* in the same second.

## Requirements

### Functional (happy path)
- Host lists a property with photos, price, calendar.
- Guest searches by location + check-in/check-out + guests.
- Guest views listing detail, books, pays.
- Host gets notified; calendar updates.

### Non-functional
- **Consistency on booking** (no double-book) > eventual consistency.
- **Read-heavy search**: 100× more searches than bookings.
- P95 search latency < 300ms.

## Back-of-envelope

- 10M listings, 100M users, 1M bookings/day.
- Search QPS peak ≈ 50k/s → must fan-out to a **search index**, not a SQL LIKE.
- Avg listing doc ≈ 2 KB → index size ≈ 20 GB (fits on a few machines).

## High-level architecture

```
  [Guest/Host]
       │
       ▼
  ┌──────────────┐
  │ API Gateway  │
  └──────┬───────┘
   ┌─────┼──────┬──────────────┬──────────────┐
   ▼     ▼      ▼              ▼              ▼
 Listing Search  Booking    Payment       Notification
 Service Service Service    Service       Service
   │      │       │           │              │
   ▼      ▼       ▼           ▼              ▼
 Postgres ES     Postgres   Stripe API     Queue
   │              (w/ row
   │               locks)
   └── CDC ──► Search index (debezium → Elasticsearch)
```

- **Search Service** talks to an Elasticsearch-style index populated via CDC from the listings DB.
- **Booking Service** owns the calendar and uses a transaction + row lock to guarantee uniqueness.
- **Payment Service** is external (Stripe) — we just track state.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.